In [1]:
import os

audio_base_path = 'audio/audio train/'
files_in_folder = os.listdir(audio_base_path) if os.path.exists(audio_base_path) else "FOLDER NOT FOUND"

print(f"Folder exists: {os.path.exists(audio_base_path)}")
print(f"Number of files in folder: {len(files_in_folder) if isinstance(files_in_folder, list) else 0}")
print(f"Example file from folder: {files_in_folder[0] if len(files_in_folder) > 0 else 'None'}")

Folder exists: True
Number of files in folder: 410
Example file from folder: .ipynb_checkpoints


In [2]:
import language_tool_python
try:
    # This line triggers the Java executable
    tool = language_tool_python.LanguageTool('en-US') 
    print(" Java and LanguageTool are working!")
except Exception as e:
    print(f"❌ ERROR: Java is likely not installed or not in PATH.")
    print(f"Details: {e}")

✅ Java and LanguageTool are working!


In [5]:
import numpy as np

# RUN THIS CELL TO OVERWRITE THE OLD FUNCTION
def extract_features_v2(audio_path):
    try:
        # Use initial_prompt to stop Whisper from "fixing" the user's grammar
        result = stt_model.transcribe(
            audio_path, 
            fp16=False, 
            initial_prompt="Umm, uhh, he go to store. I has a dream."
        )
        text = result['text']
        duration = sum([s['end'] - s['start'] for s in result['segments']])
        
        matches = tool.check(text)
        words = text.split()
        word_count = len(words)
        
        if word_count == 0: return None

        # --- ADVANCED FEATURES ---
        # Vocab Diversity: Unique words vs total words
        vocab_diversity = len(set(w.lower() for w in words)) / word_count
        
        # Word Complexity: Average length of words
        avg_word_length = sum(len(w) for w in words) / word_count
        
        # Hesitation Index: Counting pauses/commas
        hesitation_index = (text.count('...') + text.count(',')) / word_count

        return {
            "transcript": text,
            "word_count": word_count,
            "error_density": len(matches) / word_count,
            "speaking_rate": word_count / (duration / 60) if duration > 0 else 0,
            "vocab_diversity": vocab_diversity,
            "avg_word_length": avg_word_length,
            "hesitation_index": hesitation_index
        }
    except Exception as e:
        print(f"Error processing {audio_path}: {e}")
        return None

In [6]:
# 1. Setup paths
audio_base_path = 'audio/audio train/'
train_df = pd.read_csv('csv file/train.csv')
all_features_v2 = []

print(f"Processing {len(train_df)} files with V2 features...")

# 2. Run the loop
for idx, row in tqdm(train_df.iterrows(), total=len(train_df)):
    fname = str(row['filename'])
    if not fname.lower().endswith('.wav'):
        fname += '.wav'
        
    file_path = os.path.join(audio_base_path, fname)
    
    if os.path.exists(file_path):
        # We call the NEW function here
        feat = extract_features_v2(file_path) 
        
        if feat:
            # Attach the ground truth label and filename
            feat['label'] = row['label']
            feat['filename'] = row['filename']
            all_features_v2.append(feat)

# 3. Create DataFrame and Save
processed_train_v2 = pd.DataFrame(all_features_v2)

if not processed_train_v2.empty:
    processed_train_v2.to_csv('processed_train_features_v2.csv', index=False)
    print(f"\n Done! Saved {len(processed_train_v2)} records to 'processed_train_features_v2.csv'")
    print("New features included: vocab_diversity, avg_word_length, hesitation_index")
else:
    print("\n❌ No features were extracted. Ensure extract_features_v2 is defined and running.")

Processing 409 files with V2 features...


100%|██████████████████████████████████████████████████████████████████████████████| 409/409 [6:06:40<00:00, 53.79s/it]



✅ Done! Saved 409 records to 'processed_train_features_v2.csv'
New features included: vocab_diversity, avg_word_length, hesitation_index


In [11]:
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# SVR requires scaling just like Linear Regression
svr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVR(kernel='rbf', C=1.0, epsilon=0.1))
])

svr_pipeline.fit(X_train, y_train)
svr_preds = svr_pipeline.predict(X_val)

print(f"SVR MAE: {mean_absolute_error(y_val, svr_preds):.4f}")
print(f"SVR R2: {r2_score(y_val, svr_preds):.4f}")

SVR MAE: 0.5078
SVR R2: 0.1236


In [21]:
import os
import pandas as pd
from tqdm import tqdm

# Load the test list provided by the project
test_df_original = pd.read_csv('csv file/test.csv') 
audio_test_path = 'audio/audio test/'
test_features_list = []

print("Extracting features from Test Audio...")

for idx, row in tqdm(test_df_original.iterrows(), total=len(test_df_original)):
    fname = str(row['filename'])
    if not fname.lower().endswith('.wav'): 
        fname += '.wav'
    file_path = os.path.join(audio_test_path, fname)
    
    if os.path.exists(file_path):
        # Using the v2 function we defined earlier
        feat = extract_features_v2(file_path) 
        if feat:
            feat['filename'] = row['filename']
            test_features_list.append(feat)

# This creates the missing 'df_test' variable
df_test = pd.DataFrame(test_features_list)
print(f"Done! Created df_test with {len(df_test)} rows.")

Extracting features from Test Audio...


100%|██████████████████████████████████████████████████████████████████████████████| 197/197 [1:44:37<00:00, 31.87s/it]

Done! Created df_test with 197 rows.


In [22]:
# Ensure 'features' list is defined
features = ['error_density', 'word_count', 'speaking_rate', 
            'vocab_diversity', 'avg_word_length', 'hesitation_index']

# 1. Prepare Test Data
X_test = df_test[features] 

# 2. Predict using the winning SVR pipeline 
# (Make sure you use the pipeline that gave you R2: 0.1236)
df_test['label'] = svr_pipeline.predict(X_test)

# 3. Save to the required format
final_submission = df_test[['filename', 'label']]
final_submission.to_csv('final_svr_submission.csv', index=False)

print(" submission file 'final_svr_submission.csv' is ready.")

 submission file 'final_svr_submission.csv' is ready.
